# Car Evaluation: a rule-based expert system (CPU5006 S1)

This notebook walks through the whole project in order and runs it. The code it calls lives in the
`.py` modules of this repository (`data.py`, `knowledge.py`, `engine.py`, `evaluate.py`, `backward.py`,
`class_example.py`); the unit tests in `tests/` check the engine, the knowledge base and the split. Read the README for the
project summary and the "Use of generative AI" declaration.

*Notebook assembled with Claude Code from my modules and my lab notes (`notes.md`); I checked the text
and I can explain every cell.*

**The problem.** Decide how acceptable a car is (`unacc`, `acc`, `good`, `vgood`) from six things a buyer
knows, and give the reasons. **The method.** Hand-written rules with helper concepts, forward chaining,
a priority rule for conflicts, backward chaining for "why" questions, and an honest evaluation on a
held-out split.

## 1. The data

The Car Evaluation data set (Bohanec, 1997): 1,728 cars, every combination of six categorical
attributes exactly once. The labels were generated by an expert decision model (DEX), so the data is
synthetic: it tells us how well rules can recover an expert's rules.

In [1]:
import pandas as pd
from data import COLUMNS, load, split

df = load()
print(df.shape)
print(df.head())
print()
print("missing values per column:", df.isna().sum().sum(), "  duplicated rows:", df.duplicated().sum())

(1728, 7)
  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc

missing values per column: 0   duplicated rows: 0


In [2]:
# Class shares. The biggest share is the baseline: a model that always says "unacc" is right 70% of the time.
shares = df["class"].value_counts(normalize=True)
print(shares.round(3))
print()
print("baseline accuracy (always predict the majority class):", round(shares.max(), 3))

class
unacc    0.700
acc      0.222
good     0.040
vgood    0.038
Name: proportion, dtype: float64

baseline accuracy (always predict the majority class): 0.7


## 2. What the crosstabs showed

One table per attribute: for each value, the percentage of cars in each class (classes ordered worst to
best, values ordered low to high). Two values are vetoes on their own: `persons=2` and `safety=low` are
100% unacc. `vgood` needs high safety and space. The number of doors alone changes little.

Module references: looking at the data before writing rules is step 1 of Session 3 slide 23; `pd.crosstab` itself comes from the pandas documentation.

In [3]:
# from Claude Code: the two ordering lists make the tables readable (classes worst -> best, values low -> high)
ORDER = ["unacc", "acc", "good", "vgood"]
VALUE_ORDER = {
    "buying": ["low", "med", "high", "vhigh"],
    "maint": ["low", "med", "high", "vhigh"],
    "doors": ["2", "3", "4", "5more"],
    "persons": ["2", "4", "more"],
    "lug_boot": ["small", "med", "big"],
    "safety": ["low", "med", "high"],
}
for col in COLUMNS[:-1]:
    print("--- " + col + " (% of cars with that value) ---")
    table = pd.crosstab(df[col], df["class"], normalize="index") * 100
    print(table[ORDER].reindex(VALUE_ORDER[col]).round(0).astype(int))
    print()

--- buying (% of cars with that value) ---
class   unacc  acc  good  vgood
buying                         
low        60   21    11      9
med        62   27     5      6
high       75   25     0      0
vhigh      83   17     0      0

--- maint (% of cars with that value) ---
class  unacc  acc  good  vgood
maint                         
low       62   21    11      6
med       62   27     5      6
high      73   24     0      3
vhigh     83   17     0      0

--- doors (% of cars with that value) ---
class  unacc  acc  good  vgood
doors                         
2         75   19     3      2
3         69   23     4      3
4         68   24     4      5
5more     68   24     4      5

--- persons (% of cars with that value) ---
class    unacc  acc  good  vgood
persons                         
2          100    0     0      0
4           54   34     6      5
more        56   32     6      6

--- lug_boot (% of cars with that value) ---
class     unacc  acc  good  vgood
lug_boot         

In [4]:
# The two price attributes together: three combinations are 100% unacc. They become a third veto through the concept price=vhigh.
table = pd.crosstab([df["buying"], df["maint"]], df["class"], normalize="index") * 100
print(table[ORDER].round(0).astype(int))

class         unacc  acc  good  vgood
buying maint                         
high   high      67   33     0      0
       low       67   33     0      0
       med       67   33     0      0
       vhigh    100    0     0      0
low    high      57   31     0     12
       low       57    9    21     12
       med       57    9    21     12
       vhigh     67   33     0      0
med    high      67   33     0      0
       low       57    9    21     12
       med       57   31     0     12
       vhigh     67   33     0      0
vhigh  high     100    0     0      0
       low       67   33     0      0
       med       67   33     0      0
       vhigh    100    0     0      0


## 3. The split

A stratified 70/30 hold-out split with a fixed seed: 30% of *each* class goes to the test half, because
`good` and `vgood` are only 4% each and the file is sorted by attribute. The rules are designed on the
training half; the test half is scored once per version of the knowledge base.

Module references: hold-out split and no peeking, Session 3 slides 30-31.

In [5]:
train, test = split(df)
print("train rows:", len(train), " test rows:", len(test))
print()
print("class shares in train:")
print(train["class"].value_counts(normalize=True).round(3))
print()
print("class shares in test:")
print(test["class"].value_counts(normalize=True).round(3))

train rows: 1209  test rows: 519

class shares in train:
class
unacc    0.701
acc      0.222
good     0.040
vgood    0.037
Name: proportion, dtype: float64

class shares in test:
class
unacc    0.699
acc      0.222
good     0.040
vgood    0.039
Name: proportion, dtype: float64


## 4. The knowledge base

A rule is `(set_of_conditions, conclusion, reason)`. All conditions must be facts (AND); OR is written as
two rules. Conclusions that start with `car=` are final answers; the others are helper concepts
(`cheap_to_buy`, `price=low`, `family_car`, ...) that other rules build on, like the hidden PRICE and
COMFORT concepts of the original expert model.

Module references: rule format, Session 3 slide 8; car rules over attribute=value facts, slide 13; helper facts that later rules use, slides 8-9.

In [6]:
from knowledge import RULES

print(len(RULES), "rules")
print()
for number, (conditions, conclusion, reason) in enumerate(RULES, 1):
    print(str(number).rjust(2) + ". IF " + " AND ".join(sorted(conditions)) + " THEN " + conclusion)
    print("    because: " + reason)

33 rules

 1. IF persons=2 THEN car=unacc
    because: A car with only two seats is never acceptable (100% Unacc in the data).
 2. IF safety=low THEN car=unacc
    because: Low safety is a veto (100% Unacc in the data).
 3. IF buying=low THEN cheap_to_buy
    because: Low buying cost makes the car cheap to buy.
 4. IF buying=med THEN cheap_to_buy
    because: Medium buying cost is still considered affordable(ie.cheap to buy).
 5. IF maint=low THEN cheap_to_run
    because: Low maintenance costs make the car cheap to maintain.
 6. IF maint=med THEN cheap_to_run
    because: Medium maintenance costs are considered affordable to maintain.
 7. IF buying=low AND cheap_to_run THEN price=low
    because: Low buying cost combined with affordable maintenance gives a low overall price.
 8. IF buying=med AND maint=low THEN price=low
    because: Medium buying cost combined with low maintenance gives a low overall price.
 9. IF buying=med AND maint=med THEN price=med
    because: Medium buying cos

## 5. The inference engine

`row_to_facts` turns a car into six facts such as `safety=high`. `forward_chain` applies the rules again
and again until a full pass adds no new fact; each rule fires at most once and every fired rule is
recorded, which is the explanation. `verdict` resolves conflicts by priority: a veto (`car=unacc`)
always wins, otherwise the best class proved (vgood, then good, then acc); nothing proved means `unacc`.

Module references: forward chaining, Session 3 slides 10-11 and the notebook warm-up cells; conflict resolution, slides 14-15 (the class example uses first match wins, this engine uses a priority of conclusions); the majority default, slide 26.

In [7]:
# The engine itself, as it is in engine.py (the comment at the top says which parts came from Claude Code).
print(open("engine.py", encoding="utf-8").read())

# from Claude Code: the skeleton of this file (imports, function names, docstrings and the step comments).
#The code inside each function and the inline comments were written by me (Serhii), following the step comments.

# Reference: knowledge base and inference engine kept apart, Session 2 slide 21 (Components of Rule-Based Systems).
from data import COLUMNS
from knowledge import RULES


def row_to_facts(row):
    """Turn one row of the table into a set of facts like "safety=high"."""
    facts = set()

    # STEP 1: Convert each column into a fact
    for col in COLUMNS[:-1]:
        facts.add(col + "=" + str(row[col])) # Add the fact to the set

    return facts


# Reference: the forward_chain loop of Session 3 slide 10, with the fired list of slide 11 kept for the explanation.
def forward_chain(facts, rules):
    """Apply the rules again and again until no new fact appears."""
    facts = set(facts)
    fired = []
    changed = True

    # STEP 2: Apply rules until no new facts ar

In [8]:
from engine import row_to_facts, forward_chain, verdict

def explain_car(values):
    """Print the chain of reasoning for one car given as six values in the data order."""
    facts = {col + "=" + value for col, value in zip(COLUMNS[:-1], values)}
    final_facts, fired = forward_chain(facts, RULES)
    print("Car:", ", ".join(col + "=" + value for col, value in zip(COLUMNS[:-1], values)))
    for number, (conditions, conclusion, reason) in enumerate(fired, 1):
        print("  " + str(number) + ". IF " + " AND ".join(sorted(conditions)) + " THEN " + conclusion + "   (" + reason + ")")
    print("Verdict:", verdict(final_facts))
    print()

explain_car(["low", "low", "4", "4", "big", "high"])   # a cheap, safe, roomy family car
explain_car(["low", "low", "4", "2", "big", "high"])   # the same car with two seats: the veto wins

Car: buying=low, maint=low, doors=4, persons=4, lug_boot=big, safety=high
  1. IF buying=low THEN cheap_to_buy   (Low buying cost makes the car cheap to buy.)
  2. IF maint=low THEN cheap_to_run   (Low maintenance costs make the car cheap to maintain.)
  3. IF buying=low AND cheap_to_run THEN price=low   (Low buying cost combined with affordable maintenance gives a low overall price.)
  4. IF persons=4 THEN family_car   (A car with four seats can fit a family.)
  5. IF lug_boot=big THEN roomy   (Large luggage space is considered roomy.)
  6. IF family_car AND safety=high THEN car=acc   (A family car with high safety is acceptable.)
  7. IF family_car AND price=low AND safety=high THEN car=good   (An affordable family car with high safety is considered good.)
  8. IF family_car AND price=low AND roomy AND safety=high THEN car=vgood   (An affordable family car with high safety and a roomy interior is considered very good.)
Verdict: vgood

Car: buying=low, maint=low, doors=4, persons=2, l

## 6. Evaluation

Accuracy, the confusion matrix (rows = real class, columns = predicted class) and the recall of each
class, first on the training half. The error analysis lists what the wrong predictions have in common;
that is what each new version of the knowledge base was designed from.

Module references: baseline, accuracy, confusion matrix and recall, Session 3 slides 25-27 and 29; error analysis and no peeking, slides 30 and 32.

In [9]:
from evaluate import report, error_analysis

train_predicted = report("TRAIN", train)
error_analysis(train, train_predicted)


------------------ TRAIN ------------------
rows: 1209   rules: 33
accuracy: 0.963

confusion matrix (rows = real class, columns = predicted class)
predicted  unacc  acc  good  vgood
class                             
unacc        812   34     1      0
acc            0  262     7      0
good           0    0    48      0
vgood          0    1     2     42

recall per class (share of each real class that was predicted correctly)
  unacc: 0.96
  acc: 0.97
  good: 1.0
  vgood: 0.93

------------------ error analysis on TRAIN ------------------
wrong rows: 45 of 1209

how many of each kind (real class, predicted class):
class  predicted
unacc  acc          34
acc    good          7
vgood  good          2
unacc  good          1
vgood  acc           1

real acc but predicted good (7 cars). Their values:
  buying: low 5, med 2
  maint: low 4, med 3
  doors: 2 5, 3 2
  persons: 4 4, more 3
  lug_boot: med 7
  safety: med 7

real unacc but predicted acc (34 cars). Their values:
  buying: high 

The test half, scored once for the final version of the knowledge base.

In [10]:
test_predicted = report("TEST", test)


------------------ TEST ------------------
rows: 519   rules: 33
accuracy: 0.948

confusion matrix (rows = real class, columns = predicted class)
predicted  unacc  acc  good  vgood
class                             
unacc        340   21     2      0
acc            0  113     2      0
good           0    0    21      0
vgood          0    1     1     18

recall per class (share of each real class that was predicted correctly)
  unacc: 0.94
  acc: 0.98
  good: 1.0
  vgood: 0.9


### Knowledge-base log

| version | rules | train accuracy | test accuracy | what changed |
|---|---|---|---|---|
| baseline (always unacc) | 0 | 0.701 | 0.699 | |
| v1 | 19 | 0.892 | 0.882 | vetoes, price and family concepts, acc/good/vgood rules |
| v2 | 30 | 0.937 | 0.915 | a price grid (low, med, high, vhigh) instead of one price=low concept; a roomy concept; good at medium safety needs a boot of at least med; vgood also at price=med |
| v3 | 33 | 0.963 | 0.948 | grid completed for a very high buying price (vhigh+low, vhigh+med -> price=high); a high price with a small boot and medium safety is unacceptable |

## 7. Backward chaining

`prove(goal, facts, rules)` starts from a question, finds a rule that concludes it, and tries to prove
each condition the same way down to known facts, printing one line per step. Forward chaining is for
the decision; backward chaining answers one "why" question.

Module references: Session 3 slides 19-22 and the notebook warm-up cell 'Backward chaining'.

In [11]:
# The backward-chaining code, as it is in backward.py.
print(open("backward.py", encoding="utf-8").read())

# from Claude Code: this file was written in full by Claude Code on the night before the deadline (3 Nov 2026),
# when Serhii ran out of time to write the body of its skeleton himself. It adapts the backward_chain / prove
# pattern of Session 3 (slides 20-21 and the class notebook warm-up) to the three-part rules of knowledge.py.
# The rules it reasons from are Serhii's, and so is the forward-chaining engine it complements.
# Run from the project folder:  .\.venv\Scripts\python.exe backward.py
from knowledge import RULES


def prove(goal, facts, rules, depth=0):
    """Try to prove ONE goal from the facts by working backwards through the rules.

    Prints one line per step, indented by depth, so the output reads like a proof.
    Returns True if the goal is a known fact or can be derived, otherwise False."""
    indent = "    " * depth
    if goal in facts:
        print(indent + goal + ": known fact")
        return True
    for conditions, conclusion, reason in rules:
        if con

In [12]:
from backward import prove

great_car = {"buying=low", "maint=low", "doors=4", "persons=4", "lug_boot=big", "safety=high"}
print("Can car=vgood be proved for the great car?")
print("Answer:", prove("car=vgood", great_car, RULES))
print()
two_seater = (great_car - {"persons=4"}) | {"persons=2"}
print("Can car=good be proved for the two-seater?")
print("Answer:", prove("car=good", two_seater, RULES))

Can car=vgood be proved for the great car?
car=vgood: try rule needing family_car, price=low, roomy, safety=high
    family_car: try rule needing persons=4
        persons=4: known fact
    family_car: proved, because A car with four seats can fit a family.
    price=low: try rule needing buying=low, cheap_to_run
        buying=low: known fact
        cheap_to_run: try rule needing maint=low
            maint=low: known fact
        cheap_to_run: proved, because Low maintenance costs make the car cheap to maintain.
    price=low: proved, because Low buying cost combined with affordable maintenance gives a low overall price.
    roomy: try rule needing doors=4, lug_boot=med
        doors=4: known fact
        lug_boot=med: cannot be proved
    roomy: try rule needing doors=5more, lug_boot=med
        doors=5more: cannot be proved
    roomy: try rule needing lug_boot=big
        lug_boot=big: known fact
    roomy: proved, because Large luggage space is considered roomy.
    safety=high: 

## 8. Comparison: the class example

The five-rule example from the Session 3 class notebook (first match wins, default `unacc`) is scored on
the same split for scale. No single attribute could do better than the baseline here: in the crosstabs of
section 2 the most common class of every value of every attribute is `unacc`, so even OneR (Holte, 1993),
the simplest rule learner, would answer `unacc` for every car. The classes depend on combinations, which
is what the chained concepts provide.

Module references: the class example, Session 3 slides 13 and 15; a second approach to compare against, Session 2 slide 50.

In [13]:
from class_example import predict_first_match

for name, part in [("train", train), ("test", test)]:
    predicted = part.apply(predict_first_match, axis=1)
    print("class example (five rules, first match wins):", name, round((predicted == part["class"]).mean(), 3))

class example (five rules, first match wins): train 0.731
class example (five rules, first match wins): test 0.728


## 9. What it gets wrong, and limitations

See the README: the remaining mistakes, the fixed vocabulary of six attributes, maintenance by hand,
and the fact that the data is synthetic (labels generated by an expert model), which is why hand-written
rules can score so high and why the test score says little about real cars.

Module references: Session 2 slide 25 (disadvantages of rule-based systems) and slide 46 (the critical reflection).